#  AI Mock Interview Module — JobCore

**Framework name:** Weakness-Driven Dynamic Difficulty Adjustment


---
##  Uploaded CSV

Uploading interview dataset CSV. Expected columns:
- `Question Number`, `Question`, `Answer`, `Category`, `Difficulty`



In [18]:
from google.colab import files
import pandas as pd
import io

print("📂 Please upload your interview dataset CSV file...")
uploaded = files.upload()

# Get the filename from what was uploaded
filename = list(uploaded.keys())[0]

# Read into a DataFrame
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f"\n✅ Successfully loaded: {filename}")
print(f"   Rows: {len(df)} | Columns: {len(df.columns)}")

📂 Please upload your interview dataset CSV file...


Saving interview_questions.csv to interview_questions (2).csv

✅ Successfully loaded: interview_questions (2).csv
   Rows: 80 | Columns: 5


---
## Inspect Dataset

Check the shape, column names, sample rows, and value distributions.

In [19]:
import pandas as pd

print("=" * 55)
print("DATASET INSPECTION")
print("=" * 55)

print(f"\n📐 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📋 Columns: {df.columns.tolist()}")

print("\n📄 First 5 rows:")
display(df.head())

print("\n📊 Category distribution:")
print(df['Category'].value_counts().to_string())

print("\n📊 Difficulty distribution:")
print(df['Difficulty'].value_counts().to_string())

print("\n🔍 Missing values per column:")
print(df.isnull().sum().to_string())

DATASET INSPECTION

📐 Shape: 80 rows × 5 columns

📋 Columns: ['Question Number', 'Question', 'Answer', 'Category', 'Difficulty']

📄 First 5 rows:


,Question Number,Question,Answer,Category,Difficulty
0,1,What is a variable in programming?,A variable is a named storage location in memo...,Python,easy
1,2,What is the difference between a list and a tu...,A list is mutable (can be changed after creati...,Python,easy
2,3,What is a function in Python?,A function is a reusable block of code that pe...,Python,easy
3,4,What does len() do in Python?,The len() function returns the number of items...,Python,easy
4,5,What is the difference between == and is in Py...,== checks if two values are equal in content. ...,Python,easy



📊 Category distribution:
Category
Python                  14
Data Structures         11
Algorithms              10
Databases                9
OOP                      8
System Design            8
Machine Learning         7
Software Engineering     5
Operating Systems        4
Networking               4

📊 Difficulty distribution:
Difficulty
medium    35
easy      29
hard      16

🔍 Missing values per column:
Question Number    0
Question           0
Answer             0
Category           0
Difficulty         0


 Clean Dataset

Standardize column names, fix category/difficulty inconsistencies (whitespace, case, aliases), and drop duplicates.

In [20]:
print("🧹 Cleaning dataset...")

# ── Step 1: Strip whitespace from column names ──────────────────────────
df.columns = [c.strip() for c in df.columns]

# ── Step 2: Normalize Category (strip + title case) ─────────────────────
df['Category'] = df['Category'].astype(str).str.strip().str.title()

# ── Step 3: Normalize Difficulty (lowercase + alias map) ────────────────
df['Difficulty'] = df['Difficulty'].astype(str).str.strip().str.lower()

difficulty_alias_map = {
    # Easy variants
    'easy': 'easy', 'beginner': 'easy', 'basic': 'easy', 'simple': 'easy',
    'level 1': 'easy', 'l1': 'easy',
    # Medium variants
    'medium': 'medium', 'intermediate': 'medium', 'moderate': 'medium',
    'mid': 'medium', 'level 2': 'medium', 'l2': 'medium',
    # Hard variants
    'hard': 'hard', 'advanced': 'hard', 'difficult': 'hard', 'expert': 'hard',
    'complex': 'hard', 'level 3': 'hard', 'l3': 'hard',
}
df['Difficulty'] = df['Difficulty'].map(difficulty_alias_map).fillna('medium')

# ── Step 4: Drop rows with missing Question or Answer ───────────────────
before = len(df)
df = df.dropna(subset=['Question', 'Answer'])
dropped_nulls = before - len(df)

# ── Step 5: Drop duplicate questions ────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=['Question'])
dropped_dupes = before - len(df)

# ── Step 6: Reset index ──────────────────────────────────────────────────
df = df.reset_index(drop=True)

print(f"✅ Cleaning complete!")
print(f"   Dropped {dropped_nulls} rows with missing Question/Answer")
print(f"   Dropped {dropped_dupes} duplicate questions")
print(f"   Final dataset: {len(df)} rows")

print("\n📊 Category counts after cleaning:")
print(df['Category'].value_counts().to_string())

print("\n📊 Difficulty counts after cleaning:")
print(df['Difficulty'].value_counts().to_string())

🧹 Cleaning dataset...
✅ Cleaning complete!
   Dropped 0 rows with missing Question/Answer
   Dropped 0 duplicate questions
   Final dataset: 80 rows

📊 Category counts after cleaning:
Category
Python                  14
Data Structures         11
Algorithms              10
Databases                9
Oop                      8
System Design            8
Machine Learning         7
Software Engineering     5
Operating Systems        4
Networking               4

📊 Difficulty counts after cleaning:
Difficulty
medium    35
easy      29
hard      16


---
## Build Question Bank

Builded  a nested dict `question_bank[category][difficulty]` → list of question dicts.

This is the central data structure the rest of the module reads from.

In [21]:
from collections import defaultdict

# Nested defaultdict: category → difficulty → list of question dicts
question_bank = defaultdict(lambda: defaultdict(list))

for _, row in df.iterrows():
    category  = row['Category']
    difficulty = row['Difficulty']

    question_bank[category][difficulty].append({
        'question_number': row.get('Question Number', None),
        'question':   row['Question'],
        'answer':     row['Answer'],
        'category':   category,
        'difficulty': difficulty,
    })

print("✅ Question bank built!")
print(f"   Total categories: {len(question_bank)}")
print()
print("📚 Breakdown by category and difficulty:")
for cat in sorted(question_bank.keys()):
    for diff in ['easy', 'medium', 'hard']:
        n = len(question_bank[cat].get(diff, []))
        if n > 0:
            print(f"   {cat:30s} | {diff:8s} | {n} questions")

✅ Question bank built!
   Total categories: 10

📚 Breakdown by category and difficulty:
   Algorithms                     | easy     | 4 questions
   Algorithms                     | medium   | 4 questions
   Algorithms                     | hard     | 2 questions
   Data Structures                | easy     | 5 questions
   Data Structures                | medium   | 4 questions
   Data Structures                | hard     | 2 questions
   Databases                      | easy     | 3 questions
   Databases                      | medium   | 4 questions
   Databases                      | hard     | 2 questions
   Machine Learning               | easy     | 2 questions
   Machine Learning               | medium   | 4 questions
   Machine Learning               | hard     | 1 questions
   Networking                     | easy     | 2 questions
   Networking                     | medium   | 2 questions
   Oop                            | easy     | 3 questions
   Oop                     

---
##  Install Dependencies

Install `transformers`, `accelerate`, and `bitsandbytes`.



In [22]:
# Install required packages
# -q = quiet mode so output isn't overwhelming
!pip install -q transformers accelerate bitsandbytes

# Verify GPU is available
import torch
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — please switch to a GPU runtime!")
    print("   Runtime → Change runtime type → T4 GPU")

✅ GPU available: Tesla T4
   VRAM: 15.6 GB


---
##  Load Qwen Model

Loaded **Qwen/Qwen2.5-1.5B-Instruct** in **4-bit quantization** using `BitsAndBytesConfig`.



In [23]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# ── 4-bit quantization config ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load model weights in 4-bit
    bnb_4bit_use_double_quant=True,        # Nested quantization → less memory
    bnb_4bit_quant_type="nf4",             # NormalFloat4 — best quality at 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # Use fp16 for computations
)

# ── Load tokenizer ───────────────────────────────────────────────────────
print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
print("✅ Tokenizer loaded.")

# ── Load model ───────────────────────────────────────────────────────────
print("⏳ Loading model in 4-bit (this takes a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",           # Automatically place layers on GPU/CPU
    trust_remote_code=True
)
model.eval()  # Set to inference mode

print("\n✅ Qwen2.5-1.5B-Instruct loaded successfully in 4-bit!")
print(f"   Device map: {model.hf_device_map}")

# Show GPU memory used
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU memory used: {used:.2f} GB / {total:.1f} GB")

⏳ Loading tokenizer...
✅ Tokenizer loaded.
⏳ Loading model in 4-bit (this takes a few minutes)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


✅ Qwen2.5-1.5B-Instruct loaded successfully in 4-bit!
   GPU memory used: 1.88 GB / 15.6 GB


---
##  Rubric and Helper Functions

- The scoring rubric (3 dimensions, 0–5 each)
- `generate_response()` — wrapper around the Qwen model
- `extract_json()` — robust JSON parser that handles code fences and partial output

In [24]:
import json
import re

# ── Rubric definition ────────────────────────────────────────────────────
RUBRIC = {
    "technical_accuracy": {
        "description": "Is the answer technically correct, precise, and free of errors?",
        "range": "0–5"
    },
    "clarity": {
        "description": "Is the answer clearly structured, organized, and easy to follow?",
        "range": "0–5"
    },
    "depth": {
        "description": "Does the answer demonstrate deep understanding beyond surface facts?",
        "range": "0–5"
    }
}

# ── generate_response: call Qwen and return decoded text ─────────────────
def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    """
    Send a prompt to Qwen and return the generated text.

    Args:
        prompt (str): The full instruction/question prompt.
        max_new_tokens (int): Max tokens to generate (default 512).
        temperature (float): Lower = more deterministic (0.3 is good for evaluation).

    Returns:
        str: Raw model output text.
    """
    # Format as a chat message (Qwen uses the instruct template)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize and move to GPU
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = [
        out[len(inp):]
        for inp, out in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return response.strip()


# ── extract_json: parse JSON even when model adds extra text ─────────────
def extract_json(text):
    """
    Robustly extract a JSON dict from model output.
    Handles:
      - Clean JSON
      - JSON inside ```json ... ``` code fences
      - JSON buried inside other text

    Returns:
        dict or None
    """
    if not text:
        return None

    # 1. Try to extract from markdown code block
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    # 2. Try the whole text as JSON
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass

    # 3. Find the largest {...} block in the text
    # Handles cases where model adds preamble like "Here is the evaluation:"
    start = text.find('{')
    end   = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except json.JSONDecodeError:
            pass

    return None


print("✅ Rubric and helper functions defined.")
print(f"   Rubric dimensions: {list(RUBRIC.keys())}")
print(f"   Score range: 0–5 per dimension")

✅ Rubric and helper functions defined.
   Rubric dimensions: ['technical_accuracy', 'clarity', 'depth']
   Score range: 0–5 per dimension


---
## Build Evaluator

The evaluator:
1. Builds a structured prompt with the question, reference answer, and candidate answer
2. Calls Qwen and asks for a JSON-only response
3. Parses and validates the output
4. Retries up to 3 times if JSON parsing fails
5. Returns a guaranteed-safe dict even on failure

In [25]:
def build_evaluation_prompt(question, reference_answer, candidate_answer):
    """
    Build the evaluation prompt sent to Qwen.
    The prompt is highly structured to force valid JSON output.
    """
    prompt = f"""You are a strict technical interview evaluator. Your job is to score a candidate's answer.

QUESTION:
{question}

REFERENCE ANSWER (ground truth):
{reference_answer}

CANDIDATE'S ANSWER:
{candidate_answer}

SCORING RUBRIC (score each 0 to 5 as integers):
- technical_accuracy: How technically correct and precise is the answer? (0=wrong, 5=perfect)
- clarity: How clearly organized and understandable is the answer? (0=confusing, 5=crystal clear)
- depth: How deep is the understanding shown? (0=surface only, 5=expert-level insight)

INSTRUCTIONS:
- Respond with ONLY a valid JSON object. No text before or after it.
- Do not include markdown code fences.
- All string values must use double quotes.

Required JSON format:
{{
  "technical_accuracy": <integer 0-5>,
  "clarity": <integer 0-5>,
  "depth": <integer 0-5>,
  "feedback": "<one short paragraph summarizing performance>",
  "strengths": ["<specific strength 1>", "<specific strength 2>"],
  "weaknesses": ["<specific weakness 1>", "<specific weakness 2>"],
  "improved_answer": "<a concise, corrected version of the candidate's answer>"
}}"""
    return prompt


def evaluate_answer(question, reference_answer, candidate_answer, max_retries=3):
    """
    Evaluate a candidate's answer using the Qwen model and rubric.

    Args:
        question (str): The interview question.
        reference_answer (str): The correct/model answer from the dataset.
        candidate_answer (str): The candidate's actual answer.
        max_retries (int): How many times to retry on invalid JSON.

    Returns:
        dict: {
            "technical_accuracy": int,
            "clarity": int,
            "depth": int,
            "feedback": str,
            "strengths": list,
            "weaknesses": list,
            "improved_answer": str
        }
    """
    required_fields = [
        "technical_accuracy", "clarity", "depth",
        "feedback", "strengths", "weaknesses", "improved_answer"
    ]

    prompt = build_evaluation_prompt(question, reference_answer, candidate_answer)
    last_raw = ""

    for attempt in range(1, max_retries + 1):
        print(f"   🤖 Qwen generating evaluation (attempt {attempt}/{max_retries})...")
        raw_output = generate_response(prompt, max_new_tokens=600)
        last_raw = raw_output
        result = extract_json(raw_output)

        if result and all(k in result for k in required_fields):
            # Clamp scores to valid range
            for dim in ["technical_accuracy", "clarity", "depth"]:
                try:
                    result[dim] = max(0, min(5, int(result[dim])))
                except (ValueError, TypeError):
                    result[dim] = 0
            print("   ✅ Valid evaluation received.")
            return result
        else:
            print(f"   ⚠️  Attempt {attempt}: Could not parse valid JSON. Retrying...")

    # Fallback: return safe defaults so the pipeline doesn't break
    print("   ⚠️  All retries exhausted. Returning default evaluation.")
    return {
        "technical_accuracy": 0,
        "clarity": 0,
        "depth": 0,
        "feedback": "Automated evaluation failed. Please review manually.",
        "strengths": ["Unable to evaluate automatically"],
        "weaknesses": ["Unable to evaluate automatically"],
        "improved_answer": "Please provide a more detailed and structured answer.",
        "_raw_output": last_raw  # Include raw output for debugging
    }


print("✅ Evaluator function defined.")

✅ Evaluator function defined.


---
## Build Weakness Detection

Finding the lowest-scoring rubric dimension. That becomes the `weak_dimension` in the output JSON.

In [26]:
def detect_weak_dimension(scores):
    """
    Identify the rubric dimension with the lowest score.

    Args:
        scores (dict): e.g. {"technical_accuracy": 3, "clarity": 2, "depth": 4}

    Returns:
        str: Name of the weakest dimension
    """
    dimensions = ["technical_accuracy", "clarity", "depth"]
    weak_dim = min(dimensions, key=lambda d: scores.get(d, 0))
    return weak_dim


def compute_average_score(scores):
    """
    Compute the average score across all rubric dimensions.

    Args:
        scores (dict): Rubric scores dict.

    Returns:
        float: Average score.
    """
    dimensions = ["technical_accuracy", "clarity", "depth"]
    values = [scores.get(d, 0) for d in dimensions]
    return round(sum(values) / len(values), 2)


# Quick test
test_scores = {"technical_accuracy": 4, "clarity": 2, "depth": 3}
print("✅ Weakness detection defined.")
print(f"   Test scores: {test_scores}")
print(f"   → Weak dimension: {detect_weak_dimension(test_scores)}")
print(f"   → Average: {compute_average_score(test_scores)}")

✅ Weakness detection defined.
   Test scores: {'technical_accuracy': 4, 'clarity': 2, 'depth': 3}
   → Weak dimension: clarity
   → Average: 3.0


---
##Build Difficulty Adjustment

Rule-based difficulty adjuster:
- Average score ≥ 4.0 → **increase** difficulty
- Average score < 2.5 → **decrease** difficulty
- Otherwise → **keep same** difficulty

In [27]:
# Ordered list — index 0 = easiest, index 2 = hardest
DIFFICULTY_ORDER = ["easy", "medium", "hard"]


def adjust_difficulty(current_difficulty, average_score):
    """
    WDDDA — Weakness-Driven Dynamic Difficulty Adjustment.

    Rule-based logic:
      avg >= 4.0  → go up one level
      avg <  2.5  → go down one level
      otherwise   → stay same

    Difficulty is clamped to [easy, medium, hard] — cannot go
    below 'easy' or above 'hard'.

    Args:
        current_difficulty (str): Current difficulty level.
        average_score (float): Average rubric score across all dimensions.

    Returns:
        str: Next difficulty level.
    """
    # Default to medium if an unexpected value is passed
    if current_difficulty not in DIFFICULTY_ORDER:
        current_difficulty = "medium"

    current_idx = DIFFICULTY_ORDER.index(current_difficulty)

    if average_score >= 4.0:
        next_idx = min(current_idx + 1, len(DIFFICULTY_ORDER) - 1)
        reason = f"Strong performance (avg={average_score:.2f} ≥ 4.0) → increasing difficulty"
    elif average_score < 2.5:
        next_idx = max(current_idx - 1, 0)
        reason = f"Weak performance (avg={average_score:.2f} < 2.5) → decreasing difficulty"
    else:
        next_idx = current_idx
        reason = f"Average performance (2.5 ≤ avg={average_score:.2f} < 4.0) → keeping difficulty"

    next_difficulty = DIFFICULTY_ORDER[next_idx]
    print(f"   📊 {reason}")
    print(f"   🎯 Difficulty: {current_difficulty} → {next_difficulty}")
    return next_difficulty


# Quick tests
print("✅ Difficulty adjustment logic defined.")
print("\nSanity checks:")
adjust_difficulty("medium", 4.5)  # → should go to hard
adjust_difficulty("medium", 1.0)  # → should go to easy
adjust_difficulty("medium", 3.2)  # → should stay medium
adjust_difficulty("hard",   4.8)  # → should stay hard (already max)
adjust_difficulty("easy",   1.5)  # → should stay easy (already min)

✅ Difficulty adjustment logic defined.

Sanity checks:
   📊 Strong performance (avg=4.50 ≥ 4.0) → increasing difficulty
   🎯 Difficulty: medium → hard
   📊 Weak performance (avg=1.00 < 2.5) → decreasing difficulty
   🎯 Difficulty: medium → easy
   📊 Average performance (2.5 ≤ avg=3.20 < 4.0) → keeping difficulty
   🎯 Difficulty: medium → medium
   📊 Strong performance (avg=4.80 ≥ 4.0) → increasing difficulty
   🎯 Difficulty: hard → hard
   📊 Weak performance (avg=1.50 < 2.5) → decreasing difficulty
   🎯 Difficulty: easy → easy


'easy'

---
## Build Question Selection

`select_question()` picks a question from the bank:
- Respects requested category and difficulty
- Falls back to other difficulties if the exact one is empty
- Skips already-asked questions to prevent repetition

`get_next_topic()` maps the weak dimension to the next category to focus on.

In [28]:
import random


def select_question(question_bank, category=None, difficulty="medium", exclude_questions=None):
    """
    Select a question from the question bank.

    Strategy:
    1. Use requested category (random if None)
    2. Try requested difficulty first, then fall back to others
    3. Skip already-asked questions
    4. Last resort: any unused question from any category

    Args:
        question_bank (dict): {category: {difficulty: [q_dicts]}}
        category (str): Desired topic category. None = random.
        difficulty (str): Desired difficulty level.
        exclude_questions (set): Set of question strings already asked.

    Returns:
        dict or None: A question dict, or None if exhausted.
    """
    if exclude_questions is None:
        exclude_questions = set()

    available_categories = list(question_bank.keys())
    if not available_categories:
        print("⚠️  Question bank is empty!")
        return None

    # Pick a category
    if category is None or category not in question_bank:
        category = random.choice(available_categories)

    # Try requested difficulty first, then fall back gracefully
    fallback_order = [difficulty] + [d for d in DIFFICULTY_ORDER if d != difficulty]

    for diff in fallback_order:
        pool = [
            q for q in question_bank[category].get(diff, [])
            if q['question'] not in exclude_questions
        ]
        if pool:
            return random.choice(pool)

    # Absolute fallback: any category, any difficulty, not yet asked
    for cat in available_categories:
        for diff in DIFFICULTY_ORDER:
            pool = [
                q for q in question_bank[cat].get(diff, [])
                if q['question'] not in exclude_questions
            ]
            if pool:
                print(f"   ℹ️  Falling back to category={cat}, difficulty={diff}")
                return random.choice(pool)

    print("⚠️  All questions have been used!")
    return None


def get_next_topic(weak_dimension, current_category, question_bank):
    """
    Suggest the next topic to focus on, based on the weak dimension.

    Current logic: stay in the same category (target the weakness with
    a different difficulty question in the same topic area).

    Extension point: You can add a dimension→category mapping here if
    your categories align to specific skills (e.g. 'depth' → 'System Design').

    Args:
        weak_dimension (str): 'technical_accuracy', 'clarity', or 'depth'
        current_category (str): Current question's category.
        question_bank (dict): The full question bank.

    Returns:
        str: Recommended next topic/category.
    """
    # ── Extension: map weak dimensions to categories ──────────────────
    # Uncomment and customize this block once you know your category names.
    # dimension_to_category = {
    #     "technical_accuracy": "Algorithms",
    #     "clarity": "System Design",
    #     "depth": "Data Structures",
    # }
    # mapped = dimension_to_category.get(weak_dimension)
    # if mapped and mapped in question_bank:
    #     return mapped

    # Default: stay in the same category
    return current_category


print("✅ Question selection functions defined.")

✅ Question selection functions defined.


---
## Run One Demo Interview Turn


**In demo mode**, a hardcoded sample answer is used so you can see the full pipeline run without needing a real candidate.

**To use with a real candidate**, uncomment the `input()` line and remove the hardcoded answer.

In [29]:
def run_interview_turn(
    question_bank,
    category=None,
    difficulty="medium",
    candidate_answer=None,
    asked_questions=None
):
    """
    Run one complete interview turn.

    Steps:
      1. Select a question from the bank
      2. Accept/simulate a candidate answer
      3. Evaluate with Qwen + rubric
      4. Detect the weakest dimension
      5. Adjust difficulty for the next question
      6. Return structured JSON output

    Args:
        question_bank (dict): Nested question bank.
        category (str): Category to draw from. None = random.
        difficulty (str): Starting difficulty.
        candidate_answer (str): Pre-supplied answer (for demo/testing).
                                If None, the function will prompt for input.
        asked_questions (set): Questions already used in this session.

    Returns:
        dict: Full structured interview turn result.
    """
    if asked_questions is None:
        asked_questions = set()

    # ── Step 1: Select question ──────────────────────────────────────────
    print("\n🔍 Step 1: Selecting question...")
    q_data = select_question(
        question_bank,
        category=category,
        difficulty=difficulty,
        exclude_questions=asked_questions
    )
    if q_data is None:
        return None

    print(f"   Category:   {q_data['category']}")
    print(f"   Difficulty: {q_data['difficulty']}")
    print(f"   Question:   {q_data['question']}")

    # Track this question so it's not repeated
    asked_questions.add(q_data['question'])

    # ── Step 2: Get candidate answer ─────────────────────────────────────
    print("\n💬 Step 2: Candidate answer...")
    if candidate_answer is None:
        # Real mode: ask the user to type their answer
        # candidate_answer = input("\nYour answer: ")
        # Demo mode: use a sample answer
        candidate_answer = (
            "A REST API uses standard HTTP methods like GET, POST, PUT, and DELETE "
            "to interact with server-side resources identified by URLs. It is stateless, "
            "meaning each request contains all information needed to process it. "
            "I have built REST APIs using Flask and FastAPI in Python."
        )
        print(f"   [DEMO MODE] Using sample answer.")
    print(f"   Answer: {candidate_answer[:120]}...")

    # ── Step 3: Evaluate answer ──────────────────────────────────────────
    print("\n⚙️  Step 3: Evaluating with Qwen...")
    eval_result = evaluate_answer(
        question=q_data['question'],
        reference_answer=q_data['answer'],
        candidate_answer=candidate_answer
    )

    rubric_scores = {
        "technical_accuracy": eval_result["technical_accuracy"],
        "clarity":            eval_result["clarity"],
        "depth":              eval_result["depth"],
    }
    print(f"   Scores: {rubric_scores}")

    # ── Step 4: Detect weakness ──────────────────────────────────────────
    print("\n🎯 Step 4: Detecting weakest dimension...")
    weak_dim = detect_weak_dimension(rubric_scores)
    print(f"   Weakest dimension: {weak_dim}")

    # ── Step 5: Adjust difficulty ────────────────────────────────────────
    print("\n📈 Step 5: Adjusting difficulty...")
    avg_score     = compute_average_score(rubric_scores)
    next_diff     = adjust_difficulty(q_data['difficulty'], avg_score)

    # ── Step 6: Determine next topic ─────────────────────────────────────
    next_topic = get_next_topic(weak_dim, q_data['category'], question_bank)

    # ── Step 7: Assemble final output ────────────────────────────────────
    output = {
        "question":         q_data['question'],
        "reference_answer": q_data['answer'],
        "candidate_answer": candidate_answer,
        "rubric_scores":    rubric_scores,
        "feedback":         eval_result["feedback"],
        "strengths":        eval_result["strengths"],
        "weaknesses":       eval_result["weaknesses"],
        "improved_answer":  eval_result["improved_answer"],
        "weak_dimension":   weak_dim,
        "next_difficulty":  next_diff,
        "next_topic":       next_topic,
    }
    return output


print("✅ run_interview_turn() defined.")

✅ run_interview_turn() defined.


In [30]:
import json

print("=" * 60)
print("🚀  RUNNING DEMO INTERVIEW TURN")
print("=" * 60)

# Pick the first available category from the question bank
demo_category  = list(question_bank.keys())[0]
demo_difficulty = "medium"

print(f"\nDemo category:   {demo_category}")
print(f"Demo difficulty: {demo_difficulty}")

# Track questions asked in this session
session_asked = set()

# Run the turn
result = run_interview_turn(
    question_bank=question_bank,
    category=demo_category,
    difficulty=demo_difficulty,
    candidate_answer=None,   # None → demo mode uses sample answer
    asked_questions=session_asked
)

if result:
    print("\n" + "=" * 60)
    print("✅  STRUCTURED OUTPUT (JSON):")
    print("=" * 60)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print("❌  Interview turn failed — check the question bank and model.")

🚀  RUNNING DEMO INTERVIEW TURN

Demo category:   Python
Demo difficulty: medium

🔍 Step 1: Selecting question...
   Category:   Python
   Difficulty: medium
   Question:   What is a decorator in Python?

💬 Step 2: Candidate answer...
   [DEMO MODE] Using sample answer.
   Answer: A REST API uses standard HTTP methods like GET, POST, PUT, and DELETE to interact with server-side resources identified ...

⚙️  Step 3: Evaluating with Qwen...
   🤖 Qwen generating evaluation (attempt 1/3)...
   ✅ Valid evaluation received.
   Scores: {'technical_accuracy': 3, 'clarity': 4, 'depth': 2}

🎯 Step 4: Detecting weakest dimension...
   Weakest dimension: depth

📈 Step 5: Adjusting difficulty...
   📊 Average performance (2.5 ≤ avg=3.00 < 4.0) → keeping difficulty
   🎯 Difficulty: medium → medium

✅  STRUCTURED OUTPUT (JSON):
{
  "question": "What is a decorator in Python?",
  "reference_answer": "A decorator is a function that takes another function as input adds some behavior to it and returns the 

---
## Save Output

Save the interview turn result as a timestamped JSON file and download it.

In [31]:
import json
import datetime
from google.colab import files

if result:
    # Create a timestamped filename
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"interview_result_{timestamp}.json"

    # Save to disk
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f"✅ Saved: {output_filename}")

    # Download to local machine
    files.download(output_filename)
    print("📥 Download triggered.")
else:
    print("⚠️  No result to save — run the demo turn first.")

✅ Saved: interview_result_20260331_201735.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download triggered.


---
##  Annotation Template + Fine-tuning Extension

This section extends the working prototype toward **supervised fine-tuning (SFT)**.

### What fine-tuning would do

Right now, Qwen evaluates answers **zero-shot** using only the prompt. Fine-tuning would teach it
to score more consistently and align its rubric to your specific dataset's vocabulary and style.

### The three-step path to fine-tuning

1. **Generate annotation template** (this section) — extend your CSV with scoring columns
2. **Annotate** — fill in scores manually, or use the prototype to auto-generate drafts + human review
3. **Fine-tune** — use HuggingFace PEFT/LoRA on the annotated data

---

###  Create the annotation template

In [32]:
import pandas as pd

# ── New annotation columns to add ────────────────────────────────────────
annotation_columns = [
    'technical_accuracy',   # int 0-5: how correct is the answer?
    'clarity',              # int 0-5: how clear/organized?
    'depth',                # int 0-5: how deep/insightful?
    'relevance',            # int 0-5: how relevant to the question?
    'feedback',             # str: one paragraph evaluator note
    'strength_1',           # str: first identified strength
    'strength_2',           # str: second identified strength
    'weakness_1',           # str: first identified weakness
    'weakness_2',           # str: second identified weakness
    'weak_dimension',       # str: which rubric dimension is weakest
    'next_difficulty',      # str: easy/medium/hard recommendation
    'next_topic',           # str: recommended next topic
]

# Copy the original cleaned dataset
annotation_df = df.copy()

# Add empty annotation columns (to be filled manually or auto-generated)
for col in annotation_columns:
    if col not in annotation_df.columns:
        annotation_df[col] = ''  # Empty → to be filled

# Reorder columns for clarity
original_cols = [c for c in df.columns]
annotation_df = annotation_df[original_cols + annotation_columns]

# Save the template
annotation_df.to_csv('annotation_template.csv', index=False)

print("✅ Annotation template created: annotation_template.csv")
print(f"   Rows: {len(annotation_df)} | Columns: {len(annotation_df.columns)}")
print(f"   Original columns:   {original_cols}")
print(f"   Annotation columns: {annotation_columns}")

display(annotation_df.head(3))

✅ Annotation template created: annotation_template.csv
   Rows: 80 | Columns: 17
   Original columns:   ['Question Number', 'Question', 'Answer', 'Category', 'Difficulty']
   Annotation columns: ['technical_accuracy', 'clarity', 'depth', 'relevance', 'feedback', 'strength_1', 'strength_2', 'weakness_1', 'weakness_2', 'weak_dimension', 'next_difficulty', 'next_topic']


,Question Number,Question,Answer,Category,Difficulty,technical_accuracy,clarity,depth,relevance,feedback,strength_1,strength_2,weakness_1,weakness_2,weak_dimension,next_difficulty,next_topic
0,1,What is a variable in programming?,A variable is a named storage location in memo...,Python,easy,,,,,,,,,,,,
1,2,What is the difference between a list and a tu...,A list is mutable (can be changed after creati...,Python,easy,,,,,,,,,,,,
2,3,What is a function in Python?,A function is a reusable block of code that pe...,Python,easy,,,,,,,,,,,,


---
### Auto-generate annotation drafts using the prototype

Instead of annotating from scratch, use the prototype to draft scores for every row, then review and correct them manually. This is much faster than zero-shot annotation.

In [33]:
# ☢  This cell evaluates EVERY row in the dataset using Qwen.
# It can take 10–30 seconds per row depending on answer length.
# For a 100-row dataset, expect ~20 minutes.
#
# Run this to auto-generate draft annotations that you then review
# and correct before using them for fine-tuning.
#
# To run only the first N rows as a test, change the slice below.

import json
from tqdm.notebook import tqdm  # Progress bar

# How many rows to annotate. Change to len(annotation_df) for all rows.
N_ROWS = 5  # Start small to test the pipeline

print(f"   Auto-annotating first {N_ROWS} rows...")
print("   (Review and correct these before using for fine-tuning!)\n")

for i, row in tqdm(annotation_df.iloc[:N_ROWS].iterrows(), total=N_ROWS):
    # Skip rows that are already annotated
    if annotation_df.at[i, 'technical_accuracy'] != '':
        continue

    eval_result = evaluate_answer(
        question=row['Question'],
        reference_answer=row['Answer'],
        candidate_answer=row['Answer']  # Use reference as candidate → max scores
        # In real annotation, you'd have actual candidate answers here
    )

    # Write annotation fields back to the DataFrame
    annotation_df.at[i, 'technical_accuracy'] = eval_result.get('technical_accuracy', '')
    annotation_df.at[i, 'clarity']             = eval_result.get('clarity', '')
    annotation_df.at[i, 'depth']               = eval_result.get('depth', '')
    annotation_df.at[i, 'feedback']            = eval_result.get('feedback', '')

    strengths = eval_result.get('strengths', [])
    annotation_df.at[i, 'strength_1'] = strengths[0] if len(strengths) > 0 else ''
    annotation_df.at[i, 'strength_2'] = strengths[1] if len(strengths) > 1 else ''

    weaknesses = eval_result.get('weaknesses', [])
    annotation_df.at[i, 'weakness_1'] = weaknesses[0] if len(weaknesses) > 0 else ''
    annotation_df.at[i, 'weakness_2'] = weaknesses[1] if len(weaknesses) > 1 else ''

    scores = {
        'technical_accuracy': eval_result.get('technical_accuracy', 0),
        'clarity':            eval_result.get('clarity', 0),
        'depth':              eval_result.get('depth', 0),
    }
    annotation_df.at[i, 'weak_dimension']  = detect_weak_dimension(scores)
    annotation_df.at[i, 'next_difficulty'] = adjust_difficulty(
        row['Difficulty'], compute_average_score(scores)
    )
    annotation_df.at[i, 'next_topic']      = row['Category']

# Save updated template with draft annotations
annotation_df.to_csv('annotation_template_draft.csv', index=False)
print(f"\n✅ Draft annotations saved: annotation_template_draft.csv")
print("   Next step: open this CSV, review each row, and correct mistakes.")

display(annotation_df.iloc[:N_ROWS][['Question', 'technical_accuracy', 'clarity', 'depth', 'weak_dimension', 'next_difficulty']])

🤖 Auto-annotating first 5 rows...
   (Review and correct these before using for fine-tuning!)



  0%|          | 0/5 [00:00<?, ?it/s]

   🤖 Qwen generating evaluation (attempt 1/3)...
   ✅ Valid evaluation received.


IndexError: list index out of range


### Prepare data for fine-tuning

Convert the annotated CSV into a **JSONL instruction-tuning format** that works with HuggingFace TRL's `SFTTrainer`.

Each example becomes:
```
{ "instruction": "...", "input": "...", "output": "{...json...}" }
```

In [ ]:
import json
import pandas as pd

# Load the reviewed annotation file
# (Replace with annotation_template_draft.csv if you haven't reviewed yet)
try:
    annotated_df = pd.read_csv('annotation_template_draft.csv')
    print("✅ Loaded annotation_template_draft.csv")
except FileNotFoundError:
    print("⚠️  annotation_template_draft.csv not found. Run Step B first.")
    annotated_df = annotation_df  # Use in-memory version if file not found

# ── Build JSONL fine-tuning examples ─────────────────────────────────────
finetune_examples = []

for _, row in annotated_df.iterrows():
    # Skip rows that haven't been annotated
    if row.get('technical_accuracy') == '' or pd.isna(row.get('technical_accuracy')):
        continue

    instruction = (
        "You are a technical interview evaluator. "
        "Evaluate the candidate's answer against the reference answer using the rubric "
        "(technical_accuracy, clarity, depth each 0-5). "
        "Return ONLY a valid JSON object."
    )

    input_text = (
        f"QUESTION: {row['Question']}\n\n"
        f"REFERENCE ANSWER: {row['Answer']}\n\n"
        f"CANDIDATE'S ANSWER: {row['Answer']}"  # Replace with real candidate answers
    )

    output_json = {
        "technical_accuracy": int(row.get('technical_accuracy', 0)),
        "clarity":            int(row.get('clarity', 0)),
        "depth":              int(row.get('depth', 0)),
        "feedback":           str(row.get('feedback', '')),
        "strengths":          [
            str(row.get('strength_1', '')),
            str(row.get('strength_2', ''))
        ],
        "weaknesses":         [
            str(row.get('weakness_1', '')),
            str(row.get('weakness_2', ''))
        ],
        "improved_answer":    "",  # Add an improved_answer column if you annotate it
        "weak_dimension":     str(row.get('weak_dimension', '')),
        "next_difficulty":    str(row.get('next_difficulty', '')),
        "next_topic":         str(row.get('next_topic', '')),
    }

    finetune_examples.append({
        "instruction": instruction,
        "input":       input_text,
        "output":      json.dumps(output_json, ensure_ascii=False)
    })

# Save as JSONL
output_path = 'finetune_data.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for ex in finetune_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')

print(f"✅ Fine-tuning data saved: {output_path}")
print(f"   Total examples: {len(finetune_examples)}")

if finetune_examples:
    print("\nFirst example:")
    print(json.dumps(finetune_examples[0], indent=2, ensure_ascii=False)[:800] + '...')


### Fine-tuning stub


The code below is a **reference skeleton** — do not run it now. It shows the full setup so you know exactly what to add when you're ready.

```python
# ── FUTURE STAGE: LoRA fine-tuning skeleton ──────────────────────────────
# Requirements: pip install peft trl datasets

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# Load your annotated JSONL file
dataset = load_dataset('json', data_files='finetune_data.jsonl', split='train')

# LoRA config — fine-tunes only a small fraction of model weights
lora_config = LoraConfig(
    r=8,                          # Rank — higher = more capacity
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Which layers to adapt
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap base model with LoRA
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()  # Should be ~1-2% of total params

# Training arguments
training_args = TrainingArguments(
    output_dir="./qwen_interview_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
)

# SFT Trainer
trainer = SFTTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="output",  # or format with a prompt template
    args=training_args,
    max_seq_length=512,
)

trainer.train()

# Save the LoRA adapter (small file — just the delta weights)
peft_model.save_pretrained("./qwen_interview_adapter")
```

---

## Summary — What you've built

| Component | Status | File/Function |
|---|---|---|
| CSV upload + inspection | ✅ | Section 1–2 |
| Dataset cleaning | ✅ | Section 3 |
| Question bank | ✅ | `question_bank` |
| Qwen 4-bit loading | ✅ | `model`, `tokenizer` |
| Answer evaluator | ✅ | `evaluate_answer()` |
| Weakness detection | ✅ | `detect_weak_dimension()` |
| Difficulty adjustment | ✅ | `adjust_difficulty()` |
| Question selection | ✅ | `select_question()` |
| Full interview turn | ✅ | `run_interview_turn()` |
| JSON output | ✅ | `result` |
| Annotation template | ✅ | `annotation_template.csv` |
| Fine-tuning data prep | ✅ | `finetune_data.jsonl` |
| LoRA fine-tuning | 🔜 Phase 2 | See skeleton above |

### To merge into JobCore
The module's public interface is simple:
- **Input:** call `run_interview_turn(question_bank, category, difficulty, candidate_answer)`
- **Output:** a flat JSON dict matching the schema your team agreed on

No other code needs to know how the model works internally.